# Training data generation

**What this notebook does:** takes pseudonymized decisions from train and replaces the placeholders (`ОСОБА_1`, `АДРЕСА_1`, …) with fictional values. The output is in the test format (`id`, `original`, `annotations`), so it can be scored with `data/evaluate.py`.

**Steps:**
1. **Settings**: set sizes and the seed.
2. **Pool and splits**: picks documents for train / dev / test A / test B and saves the list to `data/synthetic/splits.csv`. By default the existing list is reused, so the sets do not change between runs.
3. **Generation**: for each document:
   - rules + `pymorphy3` decide the **case** of every name gap and the **gender** of every person;
   - names are taken from the reviewed table `resources/name_declensions.csv` in the right form;
   - addresses, numbers and dates are generated;
   - names that are already in clear text (judges, clerks, lawyers) are marked as `ОСОБА`.

   Takes ~20 seconds and writes `data/synthetic/{train,dev,test_a,test_b}.jsonl`. Re-run it after every change to the table.
4. **Checks**: counts, `evaluate.py` self-check, and a document to read by eye.
5. **Appendix: how the case rules were chosen**: rules vs Lapa on a gold set of real names.

All helper functions are in `synth_utils.py`. What we tried and why is in `docs/DECISION_LOG.md`; the data statistics are in `data_stats.ipynb`.

## 1. Settings

In [ ]:
import json, random
from pathlib import Path
import pandas as pd
import synth_utils as su

SEED = 42
DATA_VERSION = 2         # 1 = original generator (small name/street lists); 2 = name_lexicon.csv + mined streets/settlements
REBUILD_SPLITS = False   # True = pick the documents again (changes the sets!); False = reuse data/synthetic/splits.csv

# set sizes (in documents); used only when the splits are rebuilt
TRAIN_PER_STRATUM, DEV_PER_STRATUM, TEST_A_PER_STRATUM = 270, 20, 35   # per "proceeding type × instance" group (~16 groups)
TEST_B_PER_SUBTYPE = 50                                                 # per rare subtype
ENRICH = {"ІНФОРМАЦІЯ:email": 150, "ІНФОРМАЦІЯ:url": 150, "ІНФОРМАЦІЯ:nickname": 150, "ІНФОРМАЦІЯ:death_date": 300,
          "ІНФОРМАЦІЯ:unit_or_institution": 200, "НОМЕР:iban": 200, "НОМЕР:passport": 150, "НОМЕР:plate": 150}
NO_PII_DOCS = 500

OUT = Path("../data/synthetic"); OUT.mkdir(parents=True, exist_ok=True)
SPLITS = OUT / "splits.csv"
train = pd.read_parquet("../data/train/2024-court-decisions.parquet")
texts = train.set_index(train["id"].astype(str))["text"]

## 2. Document pool and splits
We split **by document**: each document goes into exactly one set. Rare documents are reserved first for test B, then test A, dev, and only then train.

| Set | How documents are selected |
|---|---|
| `test_b` | `TEST_B_PER_SUBTYPE` documents per rare subtype |
| `test_a` | `TEST_A_PER_STRATUM` documents from each group |
| `dev` | `DEV_PER_STRATUM` documents from each group |
| `train` | `ENRICH` enrichment + `TRAIN_PER_STRATUM` from each group + `NO_PII_DOCS` without personal data |

The pool is documents of 1,000–20,000 characters. For each one the **proceeding type × court instance** (the stratification group) and the **subtypes** of `ІНФОРМАЦІЯ`/`НОМЕР` are detected from context. Rebuilding takes 1–2 minutes.

In [ ]:
if REBUILD_SPLITS or not SPLITS.exists():
    pool = train[train["text"].str.len().between(1_000, 20_000)].sample(frac=1, random_state=SEED).reset_index(drop=True)
    pool["id"] = pool["id"].astype(str)
    pool[["proceeding", "instance"]] = pool["text"].apply(lambda t: pd.Series(su.classify(t)))
    known = (pool["proceeding"] != "unknown") & (pool["instance"] != "unknown")
    pool["stratum"] = (pool["proceeding"] + " / " + pool["instance"]).where(known, "unknown")  # all unknowns form one group
    pool["subtypes"] = pool["text"].apply(su.doc_subtypes)
    pool["has_pii"] = pool["text"].str.contains(r"(?:ОСОБА|АДРЕСА|НОМЕР|ІНФОРМАЦІЯ)_\d+")

    used, rows = set(), []
    def take(df, n, split, reason):
        ids = df.loc[~df["id"].isin(used), "id"].head(n).tolist()
        used.update(ids); rows.extend((i, split, reason) for i in ids)

    has = lambda key: pool["subtypes"].apply(lambda s: key in s)
    for key in ENRICH:
        take(pool[has(key)], TEST_B_PER_SUBTYPE, "test_b", key)
    for split, n in [("test_a", TEST_A_PER_STRATUM), ("dev", DEV_PER_STRATUM)]:
        for st, g in pool[pool["has_pii"]].groupby("stratum"):
            take(g, n, split, st)
    for key, n in ENRICH.items():
        take(pool[has(key)], n, "train", "enrich " + key)
    for st, g in pool[pool["has_pii"]].groupby("stratum"):
        take(g, TRAIN_PER_STRATUM, "train", st)
    take(pool[~pool["has_pii"]], NO_PII_DOCS, "train", "no PII")

    splits = pd.DataFrame(rows, columns=["id", "split", "reason"]).merge(pool[["id", "stratum"]], on="id")
    splits.to_csv(SPLITS, index=False)
    print(f"splits rebuilt from a pool of {len(pool)} documents")
else:
    splits = pd.read_csv(SPLITS, dtype={"id": str})
    print(f"splits loaded from {SPLITS}")
splits["split"].value_counts()

Documents per group and set. If a group has fewer documents than the quota, all available ones are taken.

In [ ]:
pd.crosstab(splits["stratum"], splits["split"], margins=True)

Rare subtypes: how many documents were collected for test B and for train enrichment.

In [ ]:
rare = splits[splits["reason"].str.contains(":")].assign(subtype=lambda d: d["reason"].str.replace("enrich ", ""))
pd.crosstab(rare["subtype"], rare["split"])

## 3. Generation
For every document: `su.annotate_rules` (case and gender) → `su.assemble_doc` (fictional values, clear-text names). Test sets use a separate half of the name lists (`name_pool="eval"`), so test names are never seen in train. Each document gets its own random generator seeded with its id, so the output is the same on every run.

**v2 (`DATA_VERSION = 2`)** uses ~2,500 surnames / ~500 first names / ~860 patronymics from `lang-uk/name_freq_dict_uk` (inflected by `pymorphy3`, `resources/name_lexicon.csv`), street and village names mined from clear-text addresses in the training corpus (`resources/streets.csv`, `resources/settlements.csv`), context-aware address shapes, more date / phone / passport formats, and company / title / institution generators for `ІНФОРМАЦІЯ`. The v1 output is kept in `data/synthetic_v1/`.

In [ ]:
if DATA_VERSION == 2:   # resources are built by scripts/build_resources.py
    tab = su.load_name_tables(Path("resources")); su.load_address_resources(Path("resources"))
else:
    tab = su.load_declensions(Path("resources/name_declensions.csv"))

records, summary = {}, []
for split in ["train", "dev", "test_a", "test_b"]:
    name_pool = "eval" if split.startswith("test") else "train"
    recs = []
    for doc_id in splits.loc[splits["split"] == split, "id"]:
        text = texts[doc_id]
        rec = su.assemble_doc(doc_id, text, tab, su.annotate_rules(text), random.Random(f"{SEED}-{doc_id}"), name_pool)
        rec["meta"]["split"] = split
        recs.append(rec)
    with open(OUT / f"{split}.jsonl", "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in recs)
    records[split] = recs
    m = pd.DataFrame([r["meta"] for r in recs])
    types = pd.Series([a["pseudonym"].split("_")[0] for r in recs for a in r["annotations"]]).value_counts()
    summary.append({"set": split, "docs": len(recs), **types.reindex(su.TYPES).fillna(0).astype(int).to_dict(),
                    "clear-text names": sum(a.get("subtype") == "clear_name" for r in recs for a in r["annotations"]),
                    "confident cases": f"{m['confident_cases'].sum() / max(m['name_gaps'].sum(), 1):.0%}"})
    print(f"{split}: {len(recs)} documents written", flush=True)
pd.DataFrame(summary).set_index("set")

## 4. Checks
**4a. Self-check with `evaluate.py`.** Each set is scored against itself. The score must be exactly 1.0: this proves the format and the offsets are right, not the quality of the data.

In [ ]:
import sys; sys.path.insert(0, "../data")
import evaluate as ev
pd.Series({s: ev.evaluate(r, r)["final_score"] for s, r in records.items()}, name="self-check score")

**4b. Subtypes** of `НОМЕР` / `ІНФОРМАЦІЯ` (all sets together).

In [ ]:
ann = pd.DataFrame([{**a, "type": a["pseudonym"].split("_")[0], "set": s}
                    for s, recs in records.items() for r in recs for a in r["annotations"]])
ann.dropna(subset=["subtype"]).groupby(["type", "subtype"]).size().rename("mentions").to_frame().T

**4c. One document to read by eye.** Generated names are shown as `[[name | ОСОБА_n]]`, other values as `[[value | label]]`. Check that the cases fit the sentence, the gender matches the verbs, and the values look natural. Change `PICK` to see another document. For a more comfortable read, run `python review_app.py`: a local page with the same highlighting and a verdict per document.

In [ ]:
PICK = 0
r = [r for r in records["dev"] if len(r["annotations"]) >= 5][PICK]
txt, out, last = r["original"], [], 0
for a in sorted(r["annotations"], key=lambda a: a["start"]):
    out += [txt[last:a["start"]], f"[[{a['entity']} | {a['pseudonym']}]]"]; last = a["end"]
print(r["id"], r["meta"])
print("".join(out + [txt[last:]])[:4000])

## 5. Appendix: how the case rules were chosen
**Gold set:** `case_eval/gold.json` holds 359 real full names from train (judges, lawyers, notaries and others whose names the registry did not hide). Their true case is known from the patronymic form. The name is replaced with `___`, and each method has to guess the case.

**Lapa** (`LapaLLM-Gemma-3-12B-instruct`) was asked in three ways; its saved answers are in `case_eval/results.json`:
- `label`: «У якому відмінку має стояти ім'я на місці пропуску? Відповідай одним словом»;
- `choice`: pick the right form from 12 variants of a stand-in name (all cases × 2 genders);
- `batch`: the same as `choice`, but 5 sentences per request.

The **rules** are re-run live, so this table changes when `rule_case` changes. «Accuracy» counts a hit when the chosen case is one of the correct ones (for masculine names the genitive and accusative look the same).

In [ ]:
gold = json.load(open("case_eval/gold.json", encoding="utf-8"))
saved = json.load(open("case_eval/results.json", encoding="utf-8"))

def rules_pick(g):
    cases, conf, _ = su.rule_case(g["left"], g["right"])
    return {"cases": [next(c for c in su.CASE_PREF if c in cases)], "confident": conf}

methods = {"rules (pymorphy3)": [rules_pick(g) for g in gold],
           "Lapa: label": saved["label"], "Lapa: choice": saved["choice"], "Lapa: batch of 5": saved["batch"]}
rows = {}
for name, res in methods.items():
    ok = pd.Series([bool(set(r["cases"]) & set(g["cases"])) for r, g in zip(res, gold)])
    row = {"accuracy": ok.mean(), "answered": pd.Series([bool(r["cases"]) for r in res]).mean()}
    if "confident" in res[0]:
        conf = pd.Series([r["confident"] for r in res])
        row.update({"share confident": conf.mean(), "accuracy where confident": ok[conf].mean()})
    by_case = pd.Series([" / ".join(g["cases"]) for g in gold])
    row.update(ok.groupby(by_case).mean().add_prefix("case: ").to_dict())
    rows[name] = row
pd.DataFrame(rows).T.map(lambda v: "—" if pd.isna(v) else f"{v:.0%}")